# ONNX YOLOv11 Bottle Cap Detection Model Validation

This notebook validates the ONNX model functionality by running object detection and visualizing results with different colors for each detected class.

In [ ]:
# Install Required Dependencies
!pip install onnxruntime opencv-python numpy matplotlib pillow

## Import Libraries and Setup
Import all required libraries and configure display settings for the notebook.

In [ ]:
import onnxruntime as ort
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
import time
from typing import List, Tuple, Dict

# Configure matplotlib for better display
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print(f"ONNXRuntime version: {ort.__version__}")
print(f"OpenCV version: {cv2.__version__}")

## Load the ONNX Model
Load the ONNX model and inspect its input/output specifications.

In [ ]:
# Model configuration
MODEL_PATH = "yolo_models/bottle_cap_detection/weights/best.onnx"
INPUT_WIDTH = 640
INPUT_HEIGHT = 640
CONFIDENCE_THRESHOLD = 0.6
NMS_THRESHOLD = 0.45

# Load ONNX model
try:
    session = ort.InferenceSession(MODEL_PATH)
    print(f"Model loaded successfully from: {MODEL_PATH}")
    
    # Get model input/output info
    input_info = session.get_inputs()[0]
    output_info = session.get_outputs()[0]
    
    print(f"\nModel Input:")
    print(f"  Name: {input_info.name}")
    print(f"  Shape: {input_info.shape}")
    print(f"  Type: {input_info.type}")
    
    print(f"\nModel Output:")
    print(f"  Name: {output_info.name}")
    print(f"  Shape: {output_info.shape}")
    print(f"  Type: {output_info.type}")
    
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please check if the model path is correct.")

## Helper Functions
Define preprocessing, inference, and visualization functions.

In [ ]:
def preprocess_image(image: np.ndarray) -> np.ndarray:
    """Preprocess image for YOLO inference."""
    # Resize image to model input size
    resized = cv2.resize(image, (INPUT_WIDTH, INPUT_HEIGHT))
    
    # Convert BGR to RGB
    rgb_image = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    
    # Normalize to [0, 1] and convert to float32
    normalized = rgb_image.astype(np.float32) / 255.0
    
    # Transpose to CHW format and add batch dimension
    input_tensor = np.transpose(normalized, (2, 0, 1))
    input_tensor = np.expand_dims(input_tensor, axis=0)
    
    return input_tensor

def run_inference(image_tensor: np.ndarray) -> np.ndarray:
    """Run inference on preprocessed image."""
    input_name = session.get_inputs()[0].name
    outputs = session.run(None, {input_name: image_tensor})
    return outputs[0]

def get_class_color(class_id: int) -> Tuple[int, int, int]:
    """Generate distinct colors for different class IDs."""
    # Use golden ratio for good color distribution
    hue = (class_id * 137.508) % 360
    
    # Convert HSV to RGB (simplified)
    c = 1.0  # Full saturation and value
    x = c * (1 - abs((hue / 60) % 2 - 1))
    m = 0.0
    
    if 0 <= hue < 60:
        r, g, b = c, x, 0
    elif 60 <= hue < 120:
        r, g, b = x, c, 0
    elif 120 <= hue < 180:
        r, g, b = 0, c, x
    elif 180 <= hue < 240:
        r, g, b = 0, x, c
    elif 240 <= hue < 300:
        r, g, b = x, 0, c
    else:
        r, g, b = c, 0, x
    
    # Convert to 0-255 range
    return (int((r + m) * 255), int((g + m) * 255), int((b + m) * 255))

def process_detections(output: np.ndarray, orig_width: int, orig_height: int) -> List[Dict]:
    """Process YOLO model outputs to extract detections."""
    detections = []
    
    # Handle 3D output: [1, 84, N] -> [N, 84]
    if len(output.shape) == 3:
        output = output[0].T  # Transpose to [N, 84]
    
    print(f"Output shape after processing: {output.shape}")
    print(f"Number of detections: {output.shape[0]}")
    
    # Calculate scale factors
    scale_x = orig_width / INPUT_WIDTH
    scale_y = orig_height / INPUT_HEIGHT
    
    # Process each detection
    for i, detection in enumerate(output):
        # Extract bbox coordinates (center format)
        x_center, y_center, width, height = detection[:4]
        
        # Find class with maximum probability (indices 4-83 are class probabilities)
        class_probs = detection[4:]
        max_class_idx = np.argmax(class_probs)
        max_prob = class_probs[max_class_idx]
        
        # Apply sigmoid to get confidence
        confidence = 1.0 / (1.0 + np.exp(-max_prob))
        
        if confidence > CONFIDENCE_THRESHOLD:
            # Scale coordinates back to original image size
            x_center_scaled = x_center * scale_x
            y_center_scaled = y_center * scale_y
            width_scaled = width * scale_x
            height_scaled = height * scale_y
            
            # Convert to corner format
            x1 = x_center_scaled - width_scaled / 2
            y1 = y_center_scaled - height_scaled / 2
            x2 = x_center_scaled + width_scaled / 2
            y2 = y_center_scaled + height_scaled / 2
            
            # Store detection info
            detections.append({
                'bbox': [x1, y1, x2, y2],
                'confidence': confidence,
                'class_id': max_class_idx,
                'raw_prob': max_prob
            })
            
            # Debug print for first few detections
            if len(detections) <= 5:
                print(f"Detection {len(detections)}: class_id={max_class_idx}, conf={confidence:.3f}, raw_prob={max_prob:.3f}")
    
    return detections

def apply_nms(detections: List[Dict], iou_threshold: float = 0.45) -> List[Dict]:
    """Apply Non-Maximum Suppression to remove overlapping detections."""
    if not detections:
        return []
    
    # Sort by confidence
    detections = sorted(detections, key=lambda x: x['confidence'], reverse=True)
    
    def calculate_iou(box1, box2):
        x1_1, y1_1, x2_1, y2_1 = box1
        x1_2, y1_2, x2_2, y2_2 = box2
        
        # Calculate intersection
        x1_inter = max(x1_1, x1_2)
        y1_inter = max(y1_1, y1_2)
        x2_inter = min(x2_1, x2_2)
        y2_inter = min(y2_1, y2_2)
        
        if x2_inter <= x1_inter or y2_inter <= y1_inter:
            return 0.0
        
        intersection = (x2_inter - x1_inter) * (y2_inter - y1_inter)
        
        # Calculate union
        area1 = (x2_1 - x1_1) * (y2_1 - y1_1)
        area2 = (x2_2 - x1_2) * (y2_2 - y1_2)
        union = area1 + area2 - intersection
        
        return intersection / union if union > 0 else 0.0
    
    # Apply NMS
    keep = []
    while detections:
        current = detections.pop(0)
        keep.append(current)
        
        # Remove detections with high IoU
        detections = [det for det in detections 
                     if calculate_iou(current['bbox'], det['bbox']) < iou_threshold]
    
    return keep

def visualize_detections(image: np.ndarray, detections: List[Dict]) -> np.ndarray:
    """Draw bounding boxes and labels on image."""
    vis_image = image.copy()
    
    for detection in detections:
        x1, y1, x2, y2 = [int(coord) for coord in detection['bbox']]
        confidence = detection['confidence']
        class_id = detection['class_id']
        
        # Get color for this class
        color = get_class_color(class_id)
        
        # Draw bounding box
        cv2.rectangle(vis_image, (x1, y1), (x2, y2), color, 2)
        
        # Draw label background
        label = f"Class{class_id}: {confidence:.2f}"
        label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)[0]
        cv2.rectangle(vis_image, (x1, y1 - label_size[1] - 10), 
                     (x1 + label_size[0], y1), color, -1)
        
        # Draw label text
        cv2.putText(vis_image, label, (x1, y1 - 5), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
    
    return vis_image

## Prepare Test Images
Load test images and prepare them for inference.

In [ ]:
# Test with a video frame
video_path = "caps.mov"

# Capture a few frames from the video
cap = cv2.VideoCapture(video_path)
test_images = []

if cap.isOpened():
    # Get frames at different positions
    frame_positions = [100, 500, 1000, 1500, 2000]
    
    for pos in frame_positions:
        cap.set(cv2.CAP_PROP_POS_FRAMES, pos)
        ret, frame = cap.read()
        if ret:
            test_images.append(frame)
            print(f"Loaded frame at position {pos}: {frame.shape}")
    
    cap.release()
    print(f"\nLoaded {len(test_images)} test images")
else:
    print(f"Error: Could not open video file {video_path}")
    
    # Fallback: try loading individual frame images if available
    frame_dir = Path("frames")
    if frame_dir.exists():
        frame_files = list(frame_dir.glob("*.jpg"))[:5]
        test_images = [cv2.imread(str(f)) for f in frame_files]
        print(f"Loaded {len(test_images)} images from frames directory")

## Run Object Detection
Execute inference on test images and extract detections.

In [ ]:
if test_images:
    # Process first image in detail
    test_image = test_images[0]
    print(f"Processing image with shape: {test_image.shape}")
    
    # Preprocess
    start_time = time.time()
    input_tensor = preprocess_image(test_image)
    preprocess_time = time.time() - start_time
    
    print(f"Input tensor shape: {input_tensor.shape}")
    print(f"Preprocessing time: {preprocess_time:.3f}s")
    
    # Run inference
    start_time = time.time()
    raw_output = run_inference(input_tensor)
    inference_time = time.time() - start_time
    
    print(f"Raw output shape: {raw_output.shape}")
    print(f"Inference time: {inference_time:.3f}s")
    
    # Process detections
    start_time = time.time()
    detections = process_detections(raw_output, test_image.shape[1], test_image.shape[0])
    processing_time = time.time() - start_time
    
    print(f"Found {len(detections)} detections before NMS")
    print(f"Detection processing time: {processing_time:.3f}s")
    
    # Apply NMS
    final_detections = apply_nms(detections, NMS_THRESHOLD)
    print(f"Found {len(final_detections)} detections after NMS")
    
    # Show class distribution
    if final_detections:
        class_counts = {}
        for det in final_detections:
            class_id = det['class_id']
            class_counts[class_id] = class_counts.get(class_id, 0) + 1
        
        print("\nClass distribution:")
        for class_id, count in sorted(class_counts.items()):
            print(f"  Class {class_id}: {count} detections")
else:
    print("No test images available!")

## Visualize Detection Results
Display the original image with bounding boxes and class labels.

In [ ]:
if test_images and 'final_detections' in locals():
    # Create visualization
    vis_image = visualize_detections(test_image, final_detections)
    
    # Display side by side
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    
    # Original image
    ax1.imshow(cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB))
    ax1.set_title('Original Image')
    ax1.axis('off')
    
    # Image with detections
    ax2.imshow(cv2.cvtColor(vis_image, cv2.COLOR_BGR2RGB))
    ax2.set_title(f'Detections (Found {len(final_detections)} objects)')
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed detection info
    print("\nDetailed Detection Results:")
    for i, det in enumerate(final_detections):
        x1, y1, x2, y2 = det['bbox']
        print(f"Detection {i+1}:")
        print(f"  Class ID: {det['class_id']}")
        print(f"  Confidence: {det['confidence']:.3f}")
        print(f"  Raw Probability: {det['raw_prob']:.3f}")
        print(f"  Bbox: ({x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f})")
        print(f"  Size: {x2-x1:.1f} x {y2-y1:.1f}")
        print()

## Test with Multiple Images
Run detection on multiple test images to validate model consistency.

In [ ]:
if len(test_images) > 1:
    # Process all test images
    all_results = []
    
    for i, image in enumerate(test_images):
        print(f"Processing image {i+1}/{len(test_images)}...")
        
        # Run inference
        input_tensor = preprocess_image(image)
        raw_output = run_inference(input_tensor)
        detections = process_detections(raw_output, image.shape[1], image.shape[0])
        final_detections = apply_nms(detections, NMS_THRESHOLD)
        
        all_results.append({
            'image_idx': i,
            'detections': final_detections,
            'detection_count': len(final_detections)
        })
        
        print(f"  Found {len(final_detections)} detections")
    
    # Display results grid
    n_images = len(test_images)
    cols = min(3, n_images)
    rows = (n_images + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows))
    if rows == 1:
        axes = [axes] if cols == 1 else axes
    else:
        axes = axes.flatten()
    
    for i, (image, result) in enumerate(zip(test_images, all_results)):
        if i < len(axes):
            vis_image = visualize_detections(image, result['detections'])
            axes[i].imshow(cv2.cvtColor(vis_image, cv2.COLOR_BGR2RGB))
            axes[i].set_title(f"Image {i+1}: {result['detection_count']} detections")
            axes[i].axis('off')
    
    # Hide unused subplots
    for j in range(len(test_images), len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    detection_counts = [r['detection_count'] for r in all_results]
    print(f"\nSummary Statistics:")
    print(f"Average detections per image: {np.mean(detection_counts):.1f}")
    print(f"Min detections: {np.min(detection_counts)}")
    print(f"Max detections: {np.max(detection_counts)}")

## Performance Analysis
Analyze model performance and create metrics to validate the ONNX model.

In [ ]:
if test_images:
    # Run performance benchmark
    print("Running performance benchmark...")
    
    times = {
        'preprocess': [],
        'inference': [],
        'postprocess': []
    }
    
    # Benchmark on multiple runs
    n_runs = 10
    for i in range(n_runs):
        image = test_images[0]  # Use first image for consistent timing
        
        # Preprocessing
        start = time.time()
        input_tensor = preprocess_image(image)
        times['preprocess'].append(time.time() - start)
        
        # Inference
        start = time.time()
        raw_output = run_inference(input_tensor)
        times['inference'].append(time.time() - start)
        
        # Postprocessing
        start = time.time()
        detections = process_detections(raw_output, image.shape[1], image.shape[0])
        final_detections = apply_nms(detections, NMS_THRESHOLD)
        times['postprocess'].append(time.time() - start)
    
    # Calculate statistics
    print(f"\nPerformance Results (averaged over {n_runs} runs):")
    for stage, time_list in times.items():
        avg_time = np.mean(time_list)
        std_time = np.std(time_list)
        print(f"{stage.capitalize()}: {avg_time:.3f}s ± {std_time:.3f}s")
    
    total_time = sum(np.mean(times[stage]) for stage in times)
    fps = 1.0 / total_time
    print(f"Total pipeline: {total_time:.3f}s")
    print(f"Estimated FPS: {fps:.1f}")
    
    # Model validation summary
    print(f"\n{'='*50}")
    print("ONNX MODEL VALIDATION SUMMARY")
    print(f"{'='*50}")
    print(f"✓ Model loaded successfully")
    print(f"✓ Input shape: {input_info.shape}")
    print(f"✓ Output shape: {output_info.shape}")
    print(f"✓ Inference working correctly")
    print(f"✓ Detections found: {len(final_detections) if 'final_detections' in locals() else 0}")
    print(f"✓ Performance: {fps:.1f} FPS")
    
    if 'all_results' in locals():
        total_detections = sum(r['detection_count'] for r in all_results)
        print(f"✓ Consistency: {total_detections} total detections across {len(all_results)} images")
    
    print(f"\n🎯 Model appears to be working correctly!")
    print(f"   Note: You can adjust CONFIDENCE_THRESHOLD ({CONFIDENCE_THRESHOLD}) to filter detections")

## Live Video Feed with Detection
Display a live feed of the caps.mov video with real-time detection visualization.

In [ ]:
def create_live_detection_display():
    """Create a live video feed with detection overlay."""
    # Open video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video {video_path}")
        return
    
    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"Video: {frame_count} frames, {fps:.1f} FPS, {width}x{height}")
    print("Press 'q' to quit, 'p' to pause/resume, 'r' to restart")
    print("Running live detection display...")
    
    frame_num = 0
    paused = False
    class_stats = {}
    
    while True:
        if not paused:
            ret, frame = cap.read()
            if not ret:
                # Loop video
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                frame_num = 0
                print("Video ended, restarting...")
                continue
            
            frame_num += 1
        
        # Run detection pipeline
        start_time = time.time()
        
        # Preprocess
        input_tensor = preprocess_image(frame)
        
        # Run inference
        raw_output = run_inference(input_tensor)
        
        # Process detections
        detections = process_detections(raw_output, frame.shape[1], frame.shape[0])
        final_detections = apply_nms(detections, NMS_THRESHOLD)
        
        # Draw detections
        vis_frame = visualize_detections(frame, final_detections)
        
        inference_time = time.time() - start_time
        
        # Update statistics
        for det in final_detections:
            class_id = det['class_id']
            class_stats[class_id] = class_stats.get(class_id, 0) + 1
        
        # Add info overlay
        info_text = [
            f"Frame: {frame_num}/{frame_count}",
            f"Detections: {len(final_detections)}",
            f"Inference: {inference_time*1000:.1f}ms",
            f"FPS: {1/inference_time:.1f}" if inference_time > 0 else "FPS: N/A"
        ]
        
        # Add class distribution
        if class_stats:
            sorted_classes = sorted(class_stats.items(), key=lambda x: x[1], reverse=True)
            info_text.append("Class distribution:")
            for class_id, count in sorted_classes[:5]:  # Show top 5 classes
                info_text.append(f"  Class{class_id}: {count}")
        
        # Draw info overlay
        y_offset = 30
        for text in info_text:
            # White background
            cv2.putText(vis_frame, text, (10, y_offset), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            # Black text
            cv2.putText(vis_frame, text, (10, y_offset), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1)
            y_offset += 25
        
        # Show pause indicator
        if paused:
            cv2.putText(vis_frame, "PAUSED", (width//2 - 50, 50), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        
        # Display frame
        cv2.imshow('ONNX YOLO Live Detection - caps.mov', vis_frame)
        
        # Handle keyboard input
        key = cv2.waitKey(1 if not paused else 0) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('p'):
            paused = not paused
            print("Paused" if paused else "Resumed")
        elif key == ord('r'):
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            frame_num = 0
            class_stats.clear()
            print("Video restarted")
        elif key == ord('s') and paused:
            # Step one frame when paused
            ret, frame = cap.read()
            if ret:
                frame_num += 1
            else:
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                frame_num = 0
    
    cap.release()
    cv2.destroyAllWindows()
    
    # Print final statistics
    print("\nFinal Class Distribution:")
    if class_stats:
        for class_id, count in sorted(class_stats.items()):
            print(f"  Class {class_id}: {count} detections")
    else:
        print("  No detections found")

# Run the live detection display
create_live_detection_display()

## Debug Output for Rust Implementation
Generate debug information to help fix the Rust code implementation.

In [ ]:
# Debug information for Rust implementation
print("=== RUST DEBUG INFORMATION ===")

if test_images and 'raw_output' in locals():
    print(f"1. Raw output shape: {raw_output.shape}")
    print(f"   - dims: {len(raw_output.shape)}")
    print(f"   - total_elements: {raw_output.size}")
    print(f"   - Expected: [1, 5, 8400] for single-class model")
    
    print(f"\n2. After transpose: {raw_output[0].T.shape}")
    print(f"   - Expected: [8400, 5]")
    
    # Show actual values for first few detections to help debug
    output_2d = raw_output[0].T  # [N, 5]
    print(f"\n3. First 3 detections (raw values):")
    for i in range(min(3, output_2d.shape[0])):
        x, y, w, h, class_prob = output_2d[i]
        confidence = 1.0 / (1.0 + np.exp(-class_prob))  # sigmoid
        print(f"   Detection {i+1}: x={x:.3f}, y={y:.3f}, w={w:.3f}, h={h:.3f}")
        print(f"                  raw_prob={class_prob:.3f}, conf={confidence:.3f}")
        if confidence > CONFIDENCE_THRESHOLD:
            print(f"                  -> ABOVE THRESHOLD ({CONFIDENCE_THRESHOLD})")
        else:
            print(f"                  -> below threshold ({CONFIDENCE_THRESHOLD})")
    
    print(f"\n4. Rust configuration should be:")
    print(f"   - features = 5 (not 84)")
    print(f"   - detected_class_id = 0 (single class)")
    print(f"   - class_prob = output_2d.at_2d::<f32>(i, 4)")
    print(f"   - confidence = 1.0 / (1.0 + (-class_prob).exp())")
    
    print(f"\n5. Expected detections with conf > {CONFIDENCE_THRESHOLD}: {len([d for d in final_detections])}")
else:
    print("No test data available for debugging")